# 09 - Deutsch-Jozsa Algorithm

Determine if a function is constant or balanced with a single quantum query.

**Concepts:** Oracle, quantum parallelism, deterministic speedup

In [ ]:
import quantsdk as qs

## Helper: Build Deutsch-Jozsa Circuit

In [ ]:
def deutsch_jozsa(n: int, oracle_type: str) -> qs.Circuit:
    """Build a Deutsch-Jozsa circuit.
    
    Args:
        n: Number of input qubits.
        oracle_type: 'constant_0', 'constant_1', or 'balanced'.
    """
    circuit = qs.Circuit(n + 1, name=f"DJ-{oracle_type}")
    
    # Initialize ancilla to |1>
    circuit.x(n)
    
    # Hadamard all qubits
    for i in range(n + 1):
        circuit.h(i)
    
    # Oracle
    circuit.barrier()
    if oracle_type == 'constant_0':
        pass  # f(x) = 0 for all x
    elif oracle_type == 'constant_1':
        circuit.x(n)  # f(x) = 1 for all x
    elif oracle_type == 'balanced':
        for i in range(n):
            circuit.cx(i, n)  # f(x) = x0 XOR x1 XOR ...
    circuit.barrier()
    
    # Hadamard input qubits
    for i in range(n):
        circuit.h(i)
    
    # Measure input qubits only
    for i in range(n):
        circuit.measure(i)
    
    return circuit

## Test with Different Oracles

In [ ]:
n = 3  # 3 input qubits

for oracle in ['constant_0', 'constant_1', 'balanced']:
    circuit = deutsch_jozsa(n, oracle)
    result = qs.run(circuit, shots=100, seed=42)
    outcome = result.most_likely
    is_constant = all(b == '0' for b in outcome)
    verdict = "CONSTANT" if is_constant else "BALANCED"
    print(f"Oracle={oracle:12s}: outcome={outcome} -> {verdict}")

print("\nAll correct with just ONE quantum query!")

## Scaling

In [ ]:
# Works for any n — still just 1 query
for n in [2, 4, 6, 8, 10]:
    circuit = deutsch_jozsa(n, 'balanced')
    result = qs.run(circuit, shots=1, seed=42)
    outcome = result.most_likely
    is_balanced = any(b == '1' for b in outcome)
    print(f"n={n:2d}: {outcome} -> {'BALANCED' if is_balanced else 'CONSTANT'} (classical needs {2**(n-1)+1} queries)")